# Gold · Dimensión de Ubicaciones

## Propósito

Construir la dimensión de ubicaciones en la capa Gold y agregar la agrupación de negocio por mercado que requiere el reporte.

## Flujo

1. Leer las ubicaciones desde la capa Silver: mercado, región, país, estado, ciudad y código postal.
2. Agregar la columna de negocio `grupo_mercado`, aplicando la regla AMERICAS = LATAM + USCA.
3. Asignar la llave sustituta y escribir la tabla.

## Destino

Capa Gold.

**Tabla:** `gld_dataco.supply_chain.desc_dim_locations`

**Llave sustituta:** `location_key`

## Regla de negocio

**AMERICAS = LATAM + USCA**

## Resultado esperado

La dimensión de ubicaciones lista para analizar los indicadores por mercado, por región y por el grupo AMERICAS.

In [0]:
# ============================================================
# GLD_DIM_LOCATIONS — Celda 1: Parámetros
# ============================================================

dbutils.widgets.text("silver_catalog", "slv_dataco", "Catálogo Silver")
dbutils.widgets.text("silver_schema", "supply_chain", "Esquema Silver")

dbutils.widgets.text("gold_catalog", "gld_dataco", "Catálogo Gold")
dbutils.widgets.text("gold_schema", "supply_chain", "Esquema Gold")

silver_catalog = dbutils.widgets.get("silver_catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_catalog = dbutils.widgets.get("gold_catalog")
gold_schema = dbutils.widgets.get("gold_schema")

SILVER_UBICACIONES_FQN = f"{silver_catalog}.{silver_schema}.md_ubicaciones"
GOLD_DIM_LOCATIONS_FQN = f"{gold_catalog}.{gold_schema}.desc_dim_locations"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_catalog}.{gold_schema}")

print(f"✓ Origen: {SILVER_UBICACIONES_FQN}")
print(f"✓ Destino: {GOLD_DIM_LOCATIONS_FQN}")

In [0]:
# ============================================================
# GLD_DIM_LOCATIONS — Celda 2: INICIO DE LOG DE EJECUCIÓN — gld_dim_locations
# ============================================================

import uuid

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control",
    key="id_ejecucion",
    debugValue=""
)

LOG_ID = str(uuid.uuid4())
NOTEBOOK = "gld_dim_locations"
CAPA = "gold"

spark.sql(f"""
INSERT INTO metadata_dataco.supply_chain.hd_etl_log
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")

In [0]:
# ============================================================
# GLD_DIM_LOCATIONS — Celda 3: Leer y remodelar desde Silver
# ============================================================

from pyspark.sql import functions as F

df_ubicaciones = spark.table(SILVER_UBICACIONES_FQN).select(
    "market",
    "order_region",
    "order_country",
    "order_state",
    "order_city",
    "order_zipcode",
    "latitude",
    "longitude"
)

# Agrupación de negocio: AMERICAS = LATAM + USCA
df_dim_locations = df_ubicaciones.withColumn(
    "grupo_mercado",
    F.when(F.col("market").isin("LATAM", "USCA"), "AMERICAS")
     .otherwise("OTROS")
)

print(f"✓ Ubicaciones únicas: {df_dim_locations.count()}")
display(df_dim_locations.limit(10))

In [0]:
# ============================================================
# GLD_DIM_LOCATIONS — Celda 4: Escritura con llave sustituta
# ============================================================

# Llave estable: hash del contenido (misma ubicación = misma llave siempre)
df_dim_locations_final = df_dim_locations.withColumn(
    "location_key",
    F.xxhash64("market", "order_region", "order_country",
               "order_state", "order_city", "order_zipcode")
)
(
    df_dim_locations_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_DIM_LOCATIONS_FQN)
)

print(f"✓ Tabla {GOLD_DIM_LOCATIONS_FQN} escrita: {df_dim_locations_final.count()} filas")

In [0]:
TABLA_DESTINO = "gld_dataco.supply_chain.desc_dim_locations"   # cambia por notebook (ver tabla)
filas_procesadas = spark.table(TABLA_DESTINO).count()

spark.sql(f"""
UPDATE metadata_dataco.supply_chain.hd_etl_log
SET estado = 'EXITOSO',
    fecha_fin = current_timestamp(),
    filas_procesadas = {filas_procesadas}
WHERE log_id = '{LOG_ID}'
""")

print(f"✓ etl_log actualizado: EXITOSO ({filas_procesadas} filas en {TABLA_DESTINO})")